**📌 Step 1 : Load the CSV file into a DataFrame**

In [ ]:
import pandas as pd
import glob
import re

# Load all CSV files from Snapshot_2026
csv_files = glob.glob("../data/Snapshot_2026/*.csv")
dfs = []
for file in csv_files:
    dfs.append(pd.read_csv(file))
df = pd.concat(dfs, ignore_index=True)

# Clean comments: Remove prefix "pseudo, YYYY-MM-DD: "
df["comment"] = df["comment"].str.split(": ", n=1).str[1]

# Rename columns
df = df.rename(columns={
    "Issue": "Root_Cause",
    "comment": "Comment"
})

# Drop rows with missing comments
df = df.dropna(subset=["Comment"])

# Drop rows with duplicates
df = df.drop_duplicates(subset=["Comment"])

# Display sample
print("Colonnes :", df.columns.tolist())
print("\nExemple de commentaires nettoyés :")
print(df["Comment"].head())

/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_81318/2310661010.py:9: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_81318/2310661010.py:9: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_81318/2310661010.py:9: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_81318/2310661010.py:9: DtypeWarning: Columns (22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(file))
/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_81318/2310661010.py:9: DtypeWarning: Columns (1,13,16,22,31,32) have mix

Colonnes : ['Snapshot_Date', 'MRP_ELEMENT_DATA', 'DELIVERY_SCHEDULE_LINE_COUNTER', 'MATERIAL_NUMBER', 'MRP_TYPE', 'SAFETY_TIME', 'GOODS_RECEIPT_PROCESSING_TIME', 'SAFETY_STOCK', 'JOINT_COMMODITY_CODE', 'PLANT', 'MRP_CONTROLLER', 'MATERIAL_DESC', 'EXCEPTION_MESSAGE_NUMBER', 'RESCHEDULING_DATE', 'VENDOR_ACCOUNT_NUMBER', 'VENDOR_NAME', 'ITEM_DELIVERY_DATE', 'PO_QTY', 'INVENTORY_QTY', 'STOCKOUT_DATE', 'BLOCKED_STOCK', 'QUALITY_INSPECTION_STOCK', 'ORDER_CONFIRMATION', 'PURCHASING_GROUP', 'SOURCING_MANAGER', 'Delay', 'Horizon', 'Total', 'MATERIAL_STATUS', 'MATERIAL_RISK', 'Comment', 'Root_Cause', 'Impact', 'Instruments_Affected']

Exemple de commentaires nettoyés :
18                                  best. LT 15.08.2025
25    on 23.03.2026 5.210pcs in quality inspection s...
34    1.620 pcs in quality inspection/to check the r...
42    Request sent out to both suppliers (Celestica ...
50     Request sent out to both suppliers (Celestica...
Name: Comment, dtype: object


**📌 Step 2 : Data Cleaning**

In [2]:
# 1. Supprimer les lignes sans commentaire
df = df.dropna(subset=["Comment"])

# 2. Supprimer les doublons de commentaires
df = df.drop_duplicates(subset=["Comment"])

# 3. Vérifier le nombre de lignes après nettoyage
print(f"Nombre de lignes après nettoyage : {len(df)}")

Nombre de lignes après nettoyage : 1610


In [3]:
# %% [Affichage des commentaires après nettoyage]

# Afficher un échantillon de commentaires
print("\n=== ÉCHANTILLON DE COMMENTAIRES ===")
print(df["Comment"].head(10))

# Afficher les commentaires avec leur contexte
print("\n=== COMMENTAIRES AVEC CONTEXTE ===")
context_df = df[["Comment", "Root_Cause", "VENDOR_NAME", "MATERIAL_NUMBER"]]
print(context_df.head(10))

# Afficher les commentaires les plus fréquents
from collections import Counter
comment_counts = Counter(df["Comment"])
print("\n=== COMMENTAIRES LES PLUS FRÉQUENTS ===")
for comment, count in comment_counts.most_common(10):
    print(f"{count}x: {comment}")

# Liste des colonnes à conserver
useful_columns = [
    'Snapshot_Date', 'MATERIAL_NUMBER', 'PLANT', 'MRP_CONTROLLER',
    'MATERIAL_DESC', 'RESCHEDULING_DATE', 'VENDOR_ACCOUNT_NUMBER',
    'VENDOR_NAME', 'ITEM_DELIVERY_DATE', 'PO_QTY', 'INVENTORY_QTY',
    'STOCKOUT_DATE', 'BLOCKED_STOCK', 'QUALITY_INSPECTION_STOCK',
    'ORDER_CONFIRMATION', 'PURCHASING_GROUP', 'SOURCING_MANAGER',
    'Delay', 'Horizon', 'Total', 'MATERIAL_STATUS', 'MATERIAL_RISK',
    'Comment', 'Root_Cause', 'Impact', 'Instruments_Affected'
]

# Sélectionner uniquement les colonnes utiles
export_df = df[useful_columns]
export_df.to_csv("../data/cleaned_comments.csv", index=False)
print("\n✅ Commentaires nettoyés exportés dans data/cleaned_comments.csv")


=== ÉCHANTILLON DE COMMENTAIRES ===
18                                  best. LT 15.08.2025
25    on 23.03.2026 5.210pcs in quality inspection s...
34    1.620 pcs in quality inspection/to check the r...
42    Request sent out to both suppliers (Celestica ...
50     Request sent out to both suppliers (Celestica...
57             No stockout - Dashboard shows wrong data
69    Supply Chain at Risk (Asteelflash & Gaudlitz) ...
75    The material is available at short notice with...
84    overall capacity problem. the problem is manag...
90    schrohm1, 2026-06-10: Request for earlier deli...
Name: Comment, dtype: object

=== COMMENTAIRES AVEC CONTEXTE ===
                                              Comment  \
18                                best. LT 15.08.2025   
25  on 23.03.2026 5.210pcs in quality inspection s...   
34  1.620 pcs in quality inspection/to check the r...   
42  Request sent out to both suppliers (Celestica ...   
50   Request sent out to both suppliers (Celestica...

**📌 Étape 3 : Tokenisation & Lemmatisation**


In [4]:
import spacy

# Charger les modèles spaCy pour l'anglais et l'allemand
nlp_en = spacy.load("en_core_web_sm")
nlp_de = spacy.load("de_core_news_sm")

# Fonction pour détecter la langue (optionnel)
from langdetect import detect

def detect_language(text):
    try:
        return detect(text)
    except:
        return "en"  # Par défaut en anglais

# Fonction pour nettoyer, tokeniser et lemmatiser
def clean_tokenize_lemmatize(text, lang="en"):
    if pd.isna(text):
        return []

    # Nettoyage de base
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Supprime les caractères spéciaux

    # Tokenisation et lemmatisation
    if lang == "en":
        doc = nlp_en(text)
    else:
        doc = nlp_de(text)

    # Retourne les lemmes (racines des mots)
    return [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

# Appliquer à la colonne Comment
df["Tokens"] = df["Comment"].apply(
    lambda x: clean_tokenize_lemmatize(x, lang=detect_language(x)) if pd.notnull(x) else []
)

# Afficher un exemple
print("\nExemple de tokens/lemmes :")
print(df["Tokens"].iloc[0])


Exemple de tokens/lemmes :
['Best', 'lt', '15082025']


📌 Step 4 : Analyse Key-words by Root-Causes

In [5]:
from collections import Counter

# 1. Extraire tous les tokens dans une liste
all_tokens = [token for tokens in df["Tokens"] for token in tokens]
token_counts = Counter(all_tokens)

# 2. Afficher les 20 mots-clés les plus fréquents
print("\nTop 20 mots-clés :")
print(token_counts.most_common(20))

# 3. (Optionnel) Afficher les mots-clés par root cause
root_cause_keywords = {}
for root_cause in df["Root_Cause"].unique():
    subset = df[df["Root_Cause"] == root_cause]
    tokens = [token for tokens in subset["Tokens"] for token in tokens]
    root_cause_keywords[root_cause] = Counter(tokens).most_common(10)

for root_cause, keywords in root_cause_keywords.items():
    print(f"\nTop mots-clés pour '{root_cause}' :")
    print(keywords)


Top 20 mots-clés :
[(' ', 899), ('\n', 552), ('lt', 222), ('stk', 206), ('eta', 189), ('Vorzug', 175), ('Klärung', 173), ('Hottlerj', 153), ('po', 119), ('Stock', 117), ('delivery', 96), ('Stück', 96), ('pc', 90), ('order', 90), ('supplier', 84), ('Woche', 81), ('date', 78), ('Material', 76), ('\n\n', 75), ('svc', 75)]

Top mots-clés pour 'nan' :
[]

Top mots-clés pour 'B_Internal Error' :
[('pull', 3), ('po', 2), ('place', 2), ('late', 2), ('purchgroup', 2), ('adjustmentdelivery', 2), ('time', 2), ('120', 2), ('dayssuppli', 2), ('confirm', 2)]

Top mots-clés pour 'S_Lack of Raw Material' :
[(' ', 152), ('\n', 58), ('lt', 38), ('Vorzug', 28), ('Woche', 27), ('1', 22), ('stk', 22), ('pc', 21), ('supplyone', 20), ('po', 19)]

Top mots-clés pour 'P_Internal Error' :
[('\n', 30), (' ', 13), ('schera12', 10), ('Bossard', 9), ('Auftragsbestätigung', 8), ('Bestellung', 8), ('Klärung', 8), ('April', 8), ('Mai', 8), ('Freigabe', 8)]

Top mots-clés pour 'S_OoS Delay' :
[('Klärung', 14), (' ', 1

**📌 Step 5 : Visualisations**

In [8]:
import seaborn as sns

# Compter les root causes par fournisseur
supplier_root_cause = df.groupby(["VENDOR_NAME", "Root_Cause"]).size().unstack(fill_value=0)

# Afficher la heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(supplier_root_cause, cmap="YlOrRd", annot=True, fmt="d")
plt.title("Root Causes par Fournisseur")
plt.xlabel("Root Cause")
plt.ylabel("VENDOR_NAME")
plt.show()

NameError: name 'plt' is not defined